# import preprocessed dataset

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split

RND = 42

split raw data into train/val/test

In [24]:
df = pd.read_csv("../data_cleaning/out/20newsgroup_preprocessed.csv")
df["text"] = df["text"].fillna("")
# labels_df = df["target"]

df_temp, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],  # maintain class dist.
    random_state=RND,
)

df_train, df_val = train_test_split(
    df_temp, test_size=0.2, stratify=df_temp["target"], random_state=RND
)

X_train = df_train["text"]
y_train = df_train["target"]

X_val = df_val["text"]
y_val = df_val["target"]

X_test = df_test["text"]
y_test = df_test["target"]

# init. vectorizer

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from vectorization.vectorize import GensimDoc2VecVectorizer, SbertVectorizer

In [ ]:
tf_vec = TfidfVectorizer(lowercase=True, stop_words="english", max_features=10_000)
# d2_vec = Doc2Vec(vector_size=1000, window=5, min_count=2, workers=4)
d2_vec = GensimDoc2VecVectorizer()
# sbert_vec = SentenceTransformer("all-MiniLM-L6-v2")
sbert_vec = SbertVectorizer()  # "all-MiniLM-L6-v2"

vectorizers = {
    "tfidf": tf_vec,
    "doc2vec": d2_vec,
    "sbert": sbert_vec,
}

# init. classifiers

In [27]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from scipy.stats import randint, uniform
from sklearn.linear_model import LogisticRegression
import numpy as np
import joblib

define search space for hyperparameters

In [ ]:
search_spaces = {
    "svm": {
        "clf": SVC(),
        "param_distributions": {
            "clf__C": uniform(0.1, 10),
            "clf__kernel": ["linear", "rbf"],
            "clf__gamma": ["scale", "auto"],
        },
    },
    "mlp": {
        "clf": MLPClassifier(max_iter=400),
        "param_distributions": {
            "clf__hidden_layer_sizes": [(50,), (100, 50)],
            "clf__activation": ["relu", "tanh"],
            "clf__alpha": uniform(1e-5, 1e-2),
            "clf__learning_rate": ["constant", "adaptive"],
        },
    },
    "dt": {
        "clf": DecisionTreeClassifier(),
        "param_distributions": {
            "clf__max_depth": randint(3, 20),
            "clf__min_samples_split": randint(2, 10),
            "clf__criterion": ["gini", "entropy"],
        },
    },
}

# random search

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RND)
best_models = {}

for vec_name, vectorizer in vectorizers.items():
    for clf_name, spec in search_spaces.items():
        name = f"{clf_name}_{vec_name}"  # e.g., svm_tfidf

        pipe = Pipeline([("vectorizer", vectorizer), ("clf", spec["clf"])])

        search = RandomizedSearchCV(
            pipe,
            param_distributions=spec["param_distributions"],
            n_iter=20,
            scoring="f1_weighted",
            cv=cv,
            random_state=RND,
            verbose=1,
            n_jobs=-1,
        )

        print(f"🔍 Running RandomizedSearchCV for {name}...")
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_

        model_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "models"))
        os.makedirs(model_dir, exist_ok=True)  # create if doesn't exist
        # save model
        joblib.dump(best_models[name], os.path.join(model_dir, f"{name}.pkl"))

        print(f"✅ Best parameters for {name}: {search.best_params_}")

# evaluation on validation set

In [ ]:
for name, model in best_models.items():
    y_pred = model.predict(X_val)
    print(f"\n📈 Evaluation report for {name.upper()} on validation set:")
    print(classification_report(y_val, y_pred))

# train clfs on train + val

In [ ]:
X_train = pd.concat([X_train, X_val], ignore_index=True)
y_train = pd.concat([y_train, y_val], ignore_index=True)